TASK-4

Scheduled End-to-End ETL Pipeline with Alerting
Goal: Orchestrate an end-to-end automated pipeline running on schedule with error alerts.
Combine extraction, transformation, and loading into an autonomous pipeline scheduled via schedule/cron that emails a summary report upon completion.

schedule / cron
logging
Python 3
Suggested Approach: Use Python schedule module or cron job, logging module for trace logs, and smtplib or webhook for automated notifications.
Deliverable: Automated pipeline executing on timer, persisting logs, and dispatching summary alert messages.

In [2]:
import logging
from datetime import datetime

# 1. Configure the logging module
# Saves log traces to 'pipeline.log' and displays them live in console
LOG_FILE = "pipeline.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler()
    ]
)

print(" Logging system initialized!")
logging.info("Test log: ETL Pipeline logging framework is active.")

 Logging system initialized!


In [3]:
import time
import requests
from bs4 import BeautifulSoup

class ETLPipeline:
    """Modular ETL Pipeline with Logging capabilities."""

    def __init__(self, base_url: str):
        self.base_url = base_url
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
        }

    def extract(self, max_pages: int = 3) -> list:
        """PHASE 1: Extract raw records across multiple pages."""
        logging.info("Starting EXTRACTION phase...")
        current_url = f"{self.base_url}/page/1/"
        raw_records = []
        page = 1

        while current_url and page <= max_pages:
            logging.info(f" Extracting Page {page}: {current_url}")
            try:
                response = requests.get(current_url, headers=self.headers, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")

                cards = soup.select("div.quote")
                for card in cards:
                    text_el = card.select_one("span.text")
                    author_el = card.select_one("small.author")
                    tag_els = card.select("a.tag")

                    raw_records.append({
                        "quote": text_el.get_text() if text_el else None,
                        "author": author_el.get_text() if author_el else None,
                        "tags": [t.get_text() for t in tag_els]
                    })

                next_btn = soup.select_one("li.next a")
                if next_btn and "href" in next_btn.attrs:
                    current_url = requests.compat.urljoin(self.base_url, next_btn["href"])
                    page += 1
                    time.sleep(1)
                else:
                    current_url = None
            except Exception as e:
                logging.error(f"Extraction failed on page {page}: {e}")
                raise

        logging.info(f"EXTRACTION COMPLETE. Captured {len(raw_records)} raw records.")
        return raw_records

    def transform(self, raw_records: list) -> list:
        """PHASE 2: Clean and format raw records into SQL-ready tuples."""
        logging.info("Starting TRANSFORMATION phase...")
        transformed_records = []

        for item in raw_records:
            # Clean string spaces, handle nulls, format list of tags into comma-separated strings
            clean_quote = item["quote"].strip() if item["quote"] else "N/A"
            clean_author = item["author"].strip() if item["author"] else "Unknown"
            clean_tags = ", ".join([t.strip() for t in item["tags"]]) if item["tags"] else "None"

            # Append as a tuple matching database schema: (quote, author, tags)
            transformed_records.append((clean_quote, clean_author, clean_tags))

        logging.info(f"TRANSFORMATION COMPLETE. Cleaned {len(transformed_records)} records.")
        return transformed_records

# --- Test Step 2 ---
pipeline = ETLPipeline(base_url="https://quotes.toscrape.com")
raw_data = pipeline.extract(max_pages=2)
clean_data = pipeline.transform(raw_data)

print(f"\nSample Cleaned Tuple for DB Loading:")
print(clean_data[0])


Sample Cleaned Tuple for DB Loading:
('“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'Albert Einstein', 'change, deep-thoughts, thinking, world')


In [4]:
import sqlite3
import time
import requests
from bs4 import BeautifulSoup

class ETLPipeline:
    """Modular ETL Pipeline with Extract, Transform, and Database Load capabilities."""

    def __init__(self, base_url: str, db_name: str = "etl_production.db"):
        self.base_url = base_url
        self.db_name = db_name
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
        }
        self._init_db()

    def _init_db(self):
        """Initializes target table schema if it doesn't exist."""
        try:
            with sqlite3.connect(self.db_name) as conn:
                cursor = conn.cursor()
                cursor.execute("""
                CREATE TABLE IF NOT EXISTS target_records (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    quote TEXT NOT NULL UNIQUE,
                    author TEXT NOT NULL,
                    tags TEXT,
                    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
                """)
                conn.commit()
            logging.info("Database schema initialized successfully.")
        except Exception as e:
            logging.error(f"Failed to initialize database: {e}")
            raise

    def extract(self, max_pages: int = 3) -> list:
        """PHASE 1: Extract raw records across multiple pages."""
        logging.info("Starting EXTRACTION phase...")
        current_url = f"{self.base_url}/page/1/"
        raw_records = []
        page = 1

        while current_url and page <= max_pages:
            try:
                response = requests.get(current_url, headers=self.headers, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")

                cards = soup.select("div.quote")
                for card in cards:
                    text_el = card.select_one("span.text")
                    author_el = card.select_one("small.author")
                    tag_els = card.select("a.tag")

                    raw_records.append({
                        "quote": text_el.get_text() if text_el else None,
                        "author": author_el.get_text() if author_el else None,
                        "tags": [t.get_text() for t in tag_els]
                    })

                next_btn = soup.select_one("li.next a")
                if next_btn and "href" in next_btn.attrs:
                    current_url = requests.compat.urljoin(self.base_url, next_btn["href"])
                    page += 1
                    time.sleep(1)
                else:
                    current_url = None
            except Exception as e:
                logging.error(f"Extraction failed on page {page}: {e}")
                raise

        logging.info(f"EXTRACTION COMPLETE. Captured {len(raw_records)} raw records.")
        return raw_records

    def transform(self, raw_records: list) -> list:
        """PHASE 2: Clean and format raw records into SQL-ready tuples."""
        logging.info("Starting TRANSFORMATION phase...")
        transformed_records = []

        for item in raw_records:
            clean_quote = item["quote"].strip() if item["quote"] else "N/A"
            clean_author = item["author"].strip() if item["author"] else "Unknown"
            clean_tags = ", ".join([t.strip() for t in item["tags"]]) if item["tags"] else "None"

            transformed_records.append((clean_quote, clean_author, clean_tags))

        logging.info(f"TRANSFORMATION COMPLETE. Cleaned {len(transformed_records)} records.")
        return transformed_records

    def load(self, clean_records: list) -> int:
        """PHASE 3: Batch loading into database with idempotent upsert."""
        logging.info("Starting LOADING phase (Batch Upsert)...")
        upsert_query = """
        INSERT INTO target_records (quote, author, tags)
        VALUES (?, ?, ?)
        ON CONFLICT(quote) DO UPDATE SET
            author = excluded.author,
            tags = excluded.tags,
            ingested_at = CURRENT_TIMESTAMP;
        """
        try:
            with sqlite3.connect(self.db_name) as conn:
                cursor = conn.cursor()
                cursor.executemany(upsert_query, clean_records)
                conn.commit()
                rows_processed = cursor.rowcount
            logging.info(f"LOADING COMPLETE. Successfully upserted {len(clean_records)} records.")
            return rows_processed
        except Exception as e:
            logging.error(f"Database loading failed: {e}")
            raise

# --- Test Step 3 ---
pipeline = ETLPipeline(base_url="https://quotes.toscrape.com")
raw_data = pipeline.extract(max_pages=2)
clean_data = pipeline.transform(raw_data)
pipeline.load(clean_data)

print("\nDatabase row count verification:")
with sqlite3.connect("etl_production.db") as conn:
    count = conn.execute("SELECT COUNT(*) FROM target_records").fetchone()[0]
    print(f"Total Rows Ingested in 'target_records' table: {count}")


Database row count verification:
Total Rows Ingested in 'target_records' table: 20


In [5]:
import logging
import sqlite3
import time
from datetime import datetime
import requests
from bs4 import BeautifulSoup

# Install schedule in Colab if not already installed
!pip install schedule -q
import schedule

# --- Configure Logging ---
LOG_FILE = "pipeline.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler()
    ]
)

# Optional Webhook URL (Leave as None for Console log alerts)
WEBHOOK_URL = None

class AutonomousETLPipeline:
    """Full End-to-End Orchestrated ETL Pipeline."""

    def __init__(self, base_url: str, db_name: str = "etl_production.db"):
        self.base_url = base_url
        self.db_name = db_name
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
        }
        self._init_db()

    def _init_db(self):
        try:
            with sqlite3.connect(self.db_name) as conn:
                cursor = conn.cursor()
                cursor.execute("""
                CREATE TABLE IF NOT EXISTS target_records (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    quote TEXT NOT NULL UNIQUE,
                    author TEXT NOT NULL,
                    tags TEXT,
                    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
                """)
                conn.commit()
            logging.info("Database schema initialized.")
        except Exception as e:
            logging.error(f"Failed to initialize DB: {e}")
            raise

    def extract(self, max_pages: int = 3) -> list:
        logging.info("Starting EXTRACTION phase...")
        current_url = f"{self.base_url}/page/1/"
        raw_records = []
        page = 1

        while current_url and page <= max_pages:
            try:
                response = requests.get(current_url, headers=self.headers, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")

                cards = soup.select("div.quote")
                for card in cards:
                    text_el = card.select_one("span.text")
                    author_el = card.select_one("small.author")
                    tag_els = card.select("a.tag")

                    raw_records.append({
                        "quote": text_el.get_text() if text_el else None,
                        "author": author_el.get_text() if author_el else None,
                        "tags": [t.get_text() for t in tag_els]
                    })

                next_btn = soup.select_one("li.next a")
                if next_btn and "href" in next_btn.attrs:
                    current_url = requests.compat.urljoin(self.base_url, next_btn["href"])
                    page += 1
                    time.sleep(1)
                else:
                    current_url = None
            except Exception as e:
                logging.error(f"Extraction error on page {page}: {e}")
                raise

        logging.info(f"EXTRACTION COMPLETE. Captured {len(raw_records)} raw records.")
        return raw_records

    def transform(self, raw_records: list) -> list:
        logging.info("Starting TRANSFORMATION phase...")
        transformed_records = []
        for item in raw_records:
            clean_quote = item["quote"].strip() if item["quote"] else "N/A"
            clean_author = item["author"].strip() if item["author"] else "Unknown"
            clean_tags = ", ".join([t.strip() for t in item["tags"]]) if item["tags"] else "None"
            transformed_records.append((clean_quote, clean_author, clean_tags))

        logging.info(f"TRANSFORMATION COMPLETE. Cleaned {len(transformed_records)} records.")
        return transformed_records

    def load(self, clean_records: list) -> int:
        logging.info("Starting LOADING phase (Batch Upsert)...")
        upsert_query = """
        INSERT INTO target_records (quote, author, tags)
        VALUES (?, ?, ?)
        ON CONFLICT(quote) DO UPDATE SET
            author = excluded.author,
            tags = excluded.tags,
            ingested_at = CURRENT_TIMESTAMP;
        """
        try:
            with sqlite3.connect(self.db_name) as conn:
                cursor = conn.cursor()
                cursor.executemany(upsert_query, clean_records)
                conn.commit()
                rows_processed = cursor.rowcount
            logging.info(f"LOADING COMPLETE. Upserted {len(clean_records)} records.")
            return rows_processed
        except Exception as e:
            logging.error(f"Database load error: {e}")
            raise

    def send_notification(self, status: str, summary: str):
        """Dispatches summary alert to Webhook or Console log."""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f" [ETL ALERT - {status}] {timestamp}\n{summary}"

        if WEBHOOK_URL:
            try:
                requests.post(WEBHOOK_URL, json={"content": message}, timeout=5)
                logging.info("Alert dispatched via Webhook.")
            except Exception as e:
                logging.warning(f"Webhook dispatch failed: {e}")
        else:
            logging.info(f"\n--- ALERT NOTIFICATION SUMMARY [{status}] ---\n{summary}\n---------------------------------------------")

    def run_pipeline(self):
        """Main orchestrator function."""
        logging.info("========== EXECUTING SCHEDULED ETL JOB ==========")
        start_time = time.time()
        try:
            raw_data = self.extract(max_pages=3)
            clean_data = self.transform(raw_data)
            self.load(clean_data)

            elapsed = round(time.time() - start_time, 2)
            summary_msg = (
                f"Status: SUCCESS\n"
                f"Extracted Records: {len(raw_data)}\n"
                f"Ingested Records:  {len(clean_data)}\n"
                f"Execution Duration: {elapsed}s"
            )
            self.send_notification("SUCCESS", summary_msg)
        except Exception as e:
            elapsed = round(time.time() - start_time, 2)
            error_msg = f"Status: FAILED\nError Details: {str(e)}\nDuration: {elapsed}s"
            self.send_notification("FAILURE", error_msg)


# --- Execution Task ---
def trigger_job():
    pipeline = AutonomousETLPipeline(base_url="https://quotes.toscrape.com")
    pipeline.run_pipeline()

# Run immediately once
trigger_job()

# Schedule to repeat (simulated for 1 loop in Colab)
schedule.every(10).seconds.do(trigger_job)

print("\nRunning 1 scheduled iteration cycle:")
schedule.run_pending()


Running 1 scheduled iteration cycle:
